In [1]:
##import libraries
import numpy as np
##pillow
from PIL import Image

##for dataset
import opendatasets as od

##pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms


In [2]:
## import data from kaggle
#{"username":"informatic","key":"8fdec244cc74352b6e476778e97056a9"}
url="https://www.kaggle.com/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals"
##download data
od.download(url)

Skipping, found downloaded files in "./animal-image-dataset-90-different-animals" (use force=True to force download)


In [3]:
##items
!ls

LICENSE
README.md
ac
animal-image-dataset-90-different-animals
animal.ipynb


In [4]:
##development constants
DATA_DIR="animal-image-dataset-90-different-animals/animals"
EPOCHS=10
IMG_SIZE=180
CHANNELS=3
VAL_SPLIT=0.10
LEARNING_RATE=0.001
TRAIN_SPLIT=0.10
MODEL_PATH="model"

In [ ]:
##model architecture
class Animal_CNN(nn.Module):
    def __init__(self, number_classess):
        super().__init__()
        ##CONVOLUTIONAL LAYERS
        self.cnn_layers=nn.Sequential(
            ##first convolutional layer
            nn.Conv2d(in_channels=CHANNELS, out_channels=16, kernel_size=(3,3),padding="same"),
            ##activation function 
            nn.ReLU(),
            ##max pooling layer
            nn.MaxPool2d(kernel_size=(2,2), stride=1),

            ##2nd convolutional layer
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3,3), padding="same"),
            ##activation function
            nn.ReLU(),
            ##Max pooling layer
            nn.MaxPool2d(kernel_size=(2,2), stride=1)
        )

        ##CLASSIFIER DENSE LAYERS
        # compute flattened feature size after two 2x2 max-pools with stride=1:
        # each such pool reduces spatial dim by 1, so final size = IMG_SIZE - 2
        conv_output_size = IMG_SIZE - 2
        ##total number of features that has to go into flatten layer
        linear_in_features = 32 * conv_output_size * conv_output_size

        self.classifier=nn.Sequential(
            nn.Flatten(),
            ##first dense layer
            nn.Linear(linear_in_features, 128),
            nn.ReLU(),
            nn.Linear(128, number_classess)
        )

    def forward(self, x):
        x = self.cnn_layers(x)
        x = self.classifier(x)
        return x

In [ ]:
##model compile function